# Importation des bibliothèques

In [1]:
import os
import sys
import findspark
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, when, regexp_replace, trim, to_date, length, concat, lit
from delta import configure_spark_with_delta_pip

In [2]:
# Configuration des variables d'environnement pour Java, Spark et Hadoop
os.environ["JAVA_HOME"] = r"C:\Users\ahoun\AppData\Local\Programs\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
os.environ["SPARK_HOME"] = r"C:\spark\spark-3.5.6-bin-hadoop3"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

findspark.init()

# Constantes

In [3]:
# chemin pour accéder aux données
FILE_PATH = '../src/data/raw/donnees_immobilieres.parquet'

# chemin pour sauvegarder les données nettoyées
OUTPUT_PATH = '../src/data/clean/donnees_immobilieres_cleaned.delta'

# Application

In [4]:
# Init SparkSession avec Delta Lake
builder = (
    SparkSession.builder
    .appName("RealStateProcessing")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "4g") 
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", "C:/tmp/spark-warehouse")
)

# On configure et on démarre
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [5]:
# chargement des données
df_raw = spark.read.parquet(FILE_PATH)

In [6]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df_raw.count(), len(df_raw.columns))

(1000, 13)

In [7]:
# Affichage du schéma
df_raw.printSchema()

root
 |-- region: string (nullable = true)
 |-- ue: string (nullable = true)
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- fonction: string (nullable = true)
 |-- designation_batiment_terrain: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- pays: string (nullable = true)
 |-- ministere: string (nullable = true)
 |-- date_inventaire: string (nullable = true)



In [8]:
# Affichage des 5 premières lignes
df_raw.show(5)

+------------------+------+-------------+--------------+-------------------+----------------------------+----+--------------------+-----------+-------------+------+--------------------+---------------+
|            region|    ue|           id|          type|           fonction|designation_batiment_terrain|dept|             adresse|code_postal|        ville|  pays|           ministere|date_inventaire|
+------------------+------+-------------+--------------+-------------------+----------------------------+----+--------------------+-----------+-------------+------+--------------------+---------------+
|            POITOU|178661|IG00100077264|Espace naturel|  Terrain en friche|        XYNTHIA17091POIRI...|  17| 21  R DU 14 JUILLET|      17230|      Charron|France|Ecologie - Dévelo...|        2017-12|
|NORD-PAS DE CALAIS|132144|IG00100007106|Espace naturel|  Terrain en friche|        QUAI AU BOIS - Du...|  59|          QU AU BOIS|      59140|    Dunkerque|France|Ecologie - Dévelo...|       

In [9]:
# suppression des doublons suivant l'id
df = df_raw.dropDuplicates(['id'])

In [10]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df.count(), len(df.columns))

(994, 13)

In [11]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
df.groupBy("id").count().filter(col("count") > 1).count()

0

In [12]:
# filtrage pour ne garder que les biens en France
df = df.filter(col("pays") == "France")

In [13]:
# affichage des pays distincts
df.select("pays").distinct().show()

+------+
|  pays|
+------+
|France|
+------+



In [14]:
# conversion de la colonne "date_invnetaire" en type date
df = df.withColumn("date_inventaire", to_date(col("date_inventaire"), "yyyy-MM"))

In [15]:
# Affichage du schéma
df.printSchema()

root
 |-- region: string (nullable = true)
 |-- ue: string (nullable = true)
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- fonction: string (nullable = true)
 |-- designation_batiment_terrain: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- pays: string (nullable = true)
 |-- ministere: string (nullable = true)
 |-- date_inventaire: date (nullable = true)



In [16]:
# mise en minuscule de la colonne "ville" pour uniformiser les données
df = df.withColumn("ville", lower(col("ville")))

In [17]:
# affichage des villes distinctes
df.select("ville").distinct().show()

+--------------------+
|               ville|
+--------------------+
|                pacé|
|   montigny-lès-metz|
|            deshaies|
|              grigny|
|            poitiers|
|        prévenchères|
|             roubaix|
|           fouesnant|
|             hillion|
|       méry-sur-oise|
|           le gosier|
|            ouangani|
|         brizambourg|
|aiguillon sur mer...|
|             contres|
|               trets|
|    athies-sous-laon|
|        bourget (le)|
|      saint-grégoire|
|              écouen|
+--------------------+
only showing top 20 rows



In [18]:
# mise en minuscule de la colonne "region" pour uniformiser les données
df = df.withColumn("region", lower(col("region")))

In [19]:
# affichage des régions distinctes
df.select("region").distinct().show()

+--------------------+
|              region|
+--------------------+
|            bretagne|
|       ile de france|
|               c.o.m|
|            limousin|
|  nord-pas de calais|
|       franche comte|
|     basse normandie|
|            auvergne|
|               d.o.m|
|           aquitaine|
|languedoc-roussillon|
|            lorraine|
|         rhone alpes|
|      midi-pyrennees|
|              centre|
|       pays de loire|
|           bourgogne|
|  champagne-ardennes|
|                paca|
|            picardie|
+--------------------+
only showing top 20 rows



In [20]:
# anonymisation des données sensibles
df = df.withColumn("ville",when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("ville")))

In [21]:
# affichage des villes distinctes
df.select("ville").distinct().show()

+--------------------+
|               ville|
+--------------------+
|                pacé|
|   montigny-lès-metz|
|            deshaies|
|              grigny|
|            poitiers|
|        prévenchères|
|             roubaix|
|           fouesnant|
|             hillion|
|       méry-sur-oise|
|           le gosier|
|            ouangani|
|         brizambourg|
|aiguillon sur mer...|
|             contres|
|               trets|
|    athies-sous-laon|
|        bourget (le)|
|      saint-grégoire|
|              écouen|
+--------------------+
only showing top 20 rows



In [22]:
# anonymisation des données sensibles
df = df.withColumn("code_postal", when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("code_postal")))

In [23]:
# affichage des codes postaux distincts
df.select("code_postal").distinct().show()

+-----------+
|code_postal|
+-----------+
|      75007|
|      86180|
|      57655|
|      14000|
|      44500|
|      13460|
|      18570|
|      53000|
|      76100|
|      05100|
|      29241|
|      34520|
|      13700|
|      59140|
|      95130|
|      95390|
|      06420|
|      77100|
|      78630|
|      17100|
+-----------+
only showing top 20 rows



In [24]:
# anonymisation des données sensibles
df = df.withColumn("adresse", when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("adresse")))

In [25]:
# affichage des adresses distinctes
df.select("adresse").distinct().show()

+--------------------+
|             adresse|
+--------------------+
|R DU MARECHAL LEC...|
|LD LE MONT FLOLENTIN|
|9  AV DOCTEUR MAU...|
|  LD LES GRIMPETEAUX|
|         LD KERMARIO|
|LD LA GREE DE LA ...|
|   LD LES PORTHENAYS|
|Lieu-dit LES SEPT...|
|         40  R DARBO|
|   50 Rue DE MALNOUE|
| 47 B R SAINT LIESNE|
|      14 R DE KEROSE|
|11  R DU ONZE NOV...|
|   LD PORT DE BRIARE|
|Cité ADMINISTRATI...|
|    23-25  BD JOFFRE|
|      BD DE L EUROPE|
|       LD LA VERNEDE|
|   Lieu-dit LE BOURG|
|          LD MERDANE|
+--------------------+
only showing top 20 rows



In [26]:
# Affichage du nombre de départements distincts
df.select("dept").distinct().count()

105

In [27]:
# Unification des codes départements : ajout d'un zéro devant les codes à un chiffre
df = df.withColumn("dept", when(length(col("dept")) == 1, concat(lit("0"), col("dept")))
    .otherwise(col("dept"))
)

In [28]:
# Affichage du nombre de départements distincts
df.select("dept").distinct().count()

105

In [29]:
# affichage du nombre de types distincts
df.select("type").distinct().count()

17

In [30]:
# Nettoyage du champ "type"
df = df.withColumn("type", regexp_replace(trim(lower(col("type"))), r'[.,;]+$', ''))

In [31]:
# Unification des types en catégories plus larges
df = df.withColumn("type", when(col("type").like("%réseau%") | col("type").like("%voirie%") | col("type").like("%voirie%"),
        "Réseaux et voirie")
    .when(col("type").like("%ouvrage%art%") | col("type").like("%ouvrage%réseau%"), "Ouvrage d'art des réseaux et voiries")
    .when(col("type").like("%enseignement%") | col("type").like("%sport%") | col("type").like("%bât%enseign%"),
        "Bâtiment d'enseignement ou de sport")
    .when(col("type").like("%culte%"), "Édifice du culte")
    .when(col("type").like("%sanitaire%") | col("type").like("%social%"), "Bâtiment sanitaire ou social")
    .when(col("type").like("%agricole%") | col("type").like("%élevage%"), "Bâtiment agricole ou d'élevage")
    .when(col("type").like("%culturel%"), "Bâtiment culturel")
    .when(col("type").like("%technique%"), "Bâtiment technique")
    .when(col("type").like("%bureau%"), "Bureau")
    .when(col("type").like("%commerce%"), "Commerce")
    .when(col("type").like("%monument%") | col("type").like("%mémorial%"), "Monument et mémorial")
    .when(col("type").like("%espace aménag%"), "Espace aménagé")
    .when(col("type").like("%espace naturel%"), "Espace naturel")
    .when(col("type").like("%intérieur%") | col("type").like("%biens du ministère%"), "Biens du ministère de l'Intérieur")
    .otherwise(col("type")
    )
)

In [32]:
# affichage du nombre de types distincts
df.select("type").distinct().count()

13

In [33]:
# affichage du nombre de ministères distincts
df.select("ministere").distinct().count()

37

In [34]:
# nettoyage de la colonne "ministere" : suppression des espaces, mise en minuscule et suppression des ponctuations finales
df = df.withColumn("ministere", regexp_replace(trim(lower(col("ministere"))), r'[.,;]+$', '')
)

In [35]:
# uniformisation des données du ministère
df = df.withColumn("ministere", when(col("ministere").like("%intérieur%"), "ministère de l'Intérieur")
    .when(col("ministere").like("%éducation%") | col("ministere").like("%education%") | 
        col("ministere").like("%enseignement sup%"), "ministère de l'Éducation Nationale")
    .when(col("ministere").like("%justice%"), "ministère de la Justice")
    .when(col("ministere").like("%santé%"), "ministère de la Santé")
    .when(col("ministere").like("%ecologie%") | col("ministere").like("%écologie%") | 
          col("ministere").like("%environnement%") | col("ministere").like("%aménagement%"), 
        "ministère de l'Écologie")
    .when(col("ministere").like("%budget%") | col("ministere").like("%comptes publics%") | 
          col("ministere").like("%fonction publique%"), "ministère du Budget")
    .when(col("ministere").like("%affaires étrangères%") | col("ministere").like("%affaires etrangeres%") | 
          col("ministere").like("%européennes%"), "ministère des Affaires Étrangères")
    .when(col("ministere").like("%economie%") | col("ministere").like("%économie%") | 
          col("ministere").like("%finances%") | col("ministere").like("%industrie%"), "ministère de l'Économie et des Finances")
    .when(col("ministere").like("%travail%") | col("ministere").like("%relations soc%") | 
          col("ministere").like("%solidarité%"), "ministère du Travail")
    .when(col("ministere").like("%culture%"), "ministère de la Culture")
    .when(col("ministere").like("%agriculture%") | col("ministere").like("%pêche%"), "ministère de l'Agriculture")
    .when(col("ministere").like("%premier ministre%") | 
          col("ministere").like("%services du premier%"), "services du Premier Ministre")
    .when(col("ministere").like("%aviation%"), "aviation civile")
    .when(col("ministere").like("%sénat%") | col("ministere").like("%senat%"), "Sénat")
    .when(col("ministere").like("%pouvoirs publics%"), "pouvoirs publics")
    .when(col("ministere").like("%multi%occupant%") | col("ministere").like("%multioccup%"), "multi-occupants")
    .when(col("ministere").like("%france domaine%") | col("ministere").like("%domaine%"), "France Domaine")
    .when(col("ministere").like("%conseil d%état%") | col("ministere").like("%conseil d'etat%"), "Conseil d'État")
    .when(col("ministere").like("%assemblée nationale%"), "Assemblée Nationale").otherwise(col("ministere"))
)

In [36]:
df.select("ministere").distinct().count()

16

In [37]:
# affichage des ministères distincts
df.select("ministere").distinct().show()

+--------------------+
|           ministere|
+--------------------+
|ministère de l'Éc...|
| Assemblée Nationale|
|ministère des Aff...|
|ministère de l'Éd...|
|ministère du Travail|
|     multi-occupants|
|services du Premi...|
|               Sénat|
|ministère de la J...|
|ministère de l'In...|
| ministère du Budget|
|      France Domaine|
|    pouvoirs publics|
|ministère de la S...|
|ministère de la C...|
|ministère de l'Éc...|
+--------------------+



In [38]:
# Affichage du schéma
df.printSchema()

root
 |-- region: string (nullable = true)
 |-- ue: string (nullable = true)
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- fonction: string (nullable = true)
 |-- designation_batiment_terrain: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- pays: string (nullable = true)
 |-- ministere: string (nullable = true)
 |-- date_inventaire: date (nullable = true)



In [39]:
# Affichage des 5 premières lignes
df.show(5)

+--------------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+-------+-----------+-----+------+--------------------+---------------+
|              region|                  ue|                  id|                type|            fonction|designation_batiment_terrain|dept|adresse|code_postal|ville|  pays|           ministere|date_inventaire|
+--------------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+-------+-----------+-----+------+--------------------+---------------+
|              centre|0976a107944e16f74...|006b81ab2cfd83b12...|Biens du ministèr...|biens du ministèr...|        biens du ministèr...|  37|   NULL|       NULL| NULL|France|ministère de l'In...|     2014-03-01|
|languedoc-roussillon|e9a9abcb36f3e5e66...|00ab869cabec0c3a8...|Biens du ministèr...|biens du ministèr...|        biens du ministèr...|  34|   NULL|       N

In [40]:
# Informations générales sur le DataFrame 
df_raw.describe().show()

+-------+-----------+--------------------+--------------------+--------------------+--------------------+----------------------------+-----------------+--------------------+-----------------+-------------+-------+-------------------+---------------+
|summary|     region|                  ue|                  id|                type|            fonction|designation_batiment_terrain|             dept|             adresse|      code_postal|        ville|   pays|          ministere|date_inventaire|
+-------+-----------+--------------------+--------------------+--------------------+--------------------+----------------------------+-----------------+--------------------+-----------------+-------------+-------+-------------------+---------------+
|  count|       1000|                 970|                1000|                1000|                1000|                        1000|             1000|                 693|              720|          720|   1000|               1000|           1000|


In [41]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df.count(), len(df.columns))

(976, 13)

In [42]:
# Création du dossier si il n'existe pas
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

In [43]:
# Sauvegarde des données nettoyées au format delta
df.write.format("delta").mode("overwrite").save(OUTPUT_PATH)